In [33]:
!pip install ipywidgets
!pip install folium
!pip install geopandas
!pip install shapely
!pip install pandas
!pip install requests
!pip install ipyleaflet


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [51]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
import datetime
import requests
import math

# Define static shapefile paths
LANDSAT_SHAPEFILE = 'shapefiles/WRS2_descending.shp'
SENTINEL_SHAPEFILE = 'shapefiles/S2A_relative_orbit_groundtrack_10Sec.shp'

# Function to check if a location is within satellite orbit
def is_location_in_orbit(lon, lat, shapefile_path):
    gdf = gpd.read_file(shapefile_path)
    point = Point(lon, lat)
    return any(gdf.contains(point))

# Function to calculate overpass times based on orbital parameters
def calculate_overpass_times(lat, lon, start_date, end_date, repeat_cycle, equatorial_crossing_time):
    start_date = datetime.datetime.strptime(start_date, '%Y-%m-%d')
    end_date = datetime.datetime.strptime(end_date, '%Y-%m-%d')
    delta = datetime.timedelta(days=repeat_cycle)
    
    overpass_times = []
    current_date = start_date
    while current_date <= end_date:
        overpass_time = current_date + datetime.timedelta(hours=equatorial_crossing_time)
        overpass_times.append(overpass_time.strftime('%Y-%m-%d %H:%M:%S'))
        current_date += delta
    
    return overpass_times

# Function to get overpass times for both Landsat 8 and Sentinel-2
def get_satellite_overpass(lat, lon, start_date, end_date):
    landsat_overpass = calculate_overpass_times(lat, lon, start_date, end_date, repeat_cycle=16, equatorial_crossing_time=10)
    sentinel_overpass = calculate_overpass_times(lat, lon, start_date, end_date, repeat_cycle=5, equatorial_crossing_time=10.5)
    
    landsat_data = pd.DataFrame({'date': landsat_overpass, 'Satellite': 'Landsat 8'})
    sentinel_data = pd.DataFrame({'date': sentinel_overpass, 'Satellite': 'Sentinel-2'})
    
    combined_data = pd.concat([landsat_data, sentinel_data], ignore_index=True)
    return combined_data

# Function to fetch weather data
def get_weather_data(lat, lon, date):
    url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M,CLOUD_AMT,PRECTOTCORR&community=RE&longitude={lon}&latitude={lat}&start={date}&end={date}&format=JSON"
    response = requests.get(url).json()
    if 'properties' in response:
        data = response['properties']['parameter']
        return {
            "Temperature (°C)": data['T2M'][date],
            "Cloud Cover (%)": data['CLOUD_AMT'][date],
            "Precipitation (mm)": data['PRECTOTCORR'][date]
        }
    else:
        return "No Weather Data Found"

# Function to get satellite overpass and weather data
def get_satellite_info(lat, lon, start_date, end_date):
    landsat_in_orbit = is_location_in_orbit(lon, lat, LANDSAT_SHAPEFILE)
    sentinel_in_orbit = is_location_in_orbit(lon, lat, SENTINEL_SHAPEFILE)
    
    if landsat_in_orbit or sentinel_in_orbit:
        overpass_times = get_satellite_overpass(lat, lon, start_date, end_date)
        weather_data = get_weather_data(lat, lon, start_date)
        return overpass_times, weather_data
    else:
        return "Location not in satellite orbit"

# Example usage
lat = 27.7
lon = 85.3
start_date = '2024-03-30'
end_date = '2024-04-30'

result = get_satellite_info(lat, lon, start_date, end_date)
print(result)


(                  date   Satellite
0  2024-03-30 10:00:00   Landsat 8
1  2024-04-15 10:00:00   Landsat 8
2  2024-03-30 10:30:00  Sentinel-2
3  2024-04-04 10:30:00  Sentinel-2
4  2024-04-09 10:30:00  Sentinel-2
5  2024-04-14 10:30:00  Sentinel-2
6  2024-04-19 10:30:00  Sentinel-2
7  2024-04-24 10:30:00  Sentinel-2
8  2024-04-29 10:30:00  Sentinel-2, 'No Weather Data Found')


In [48]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
import datetime
import requests
from ipyleaflet import Map, Marker, basemaps, basemap_to_tiles
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Output

# Define static shapefile paths
LANDSAT_SHAPEFILE = 'shapefiles/WRS2_descending.shp'
SENTINEL_SHAPEFILE = 'shapefiles/S2A_relative_orbit_groundtrack_10Sec.shp'

# Function to check if a location is within satellite orbit
def is_location_in_orbit(lon, lat, shapefile_path):
    gdf = gpd.read_file(shapefile_path)
    point = Point(lon, lat)
    return any(gdf.contains(point))

# Function to calculate overpass times based on orbital parameters
def calculate_overpass_times(lat, lon, start_date, end_date, repeat_cycle, equatorial_crossing_time):
    start_date = datetime.datetime.strptime(start_date, '%Y-%m-%d')
    end_date = datetime.datetime.strptime(end_date, '%Y-%m-%d')
    delta = datetime.timedelta(days=repeat_cycle)
    
    overpass_times = []
    current_date = start_date
    while current_date <= end_date:
        overpass_time = current_date + datetime.timedelta(hours=equatorial_crossing_time)
        overpass_times.append(overpass_time.strftime('%Y-%m-%d %H:%M:%S'))
        current_date += delta
    
    return overpass_times

# Function to get overpass times for both Landsat 8 and Sentinel-2
def get_satellite_overpass(lat, lon, start_date, end_date):
    landsat_overpass = calculate_overpass_times(lat, lon, start_date, end_date, repeat_cycle=16, equatorial_crossing_time=10)
    sentinel_overpass = calculate_overpass_times(lat, lon, start_date, end_date, repeat_cycle=5, equatorial_crossing_time=10.5)
    
    landsat_data = pd.DataFrame({'date': landsat_overpass, 'Satellite': 'Landsat 8'})
    sentinel_data = pd.DataFrame({'date': sentinel_overpass, 'Satellite': 'Sentinel-2'})
    
    combined_data = pd.concat([landsat_data, sentinel_data], ignore_index=True)
    return combined_data

# Function to fetch weather data
def get_weather_data(lat, lon, date):
    url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M,CLOUD_AMT,PRECTOTCORR&community=RE&longitude={lon}&latitude={lat}&start={date}&end={date}&format=JSON"
    response = requests.get(url).json()
    if 'properties' in response:
        data = response['properties']['parameter']
        return {
            "Temperature (°C)": data['T2M'][date],
            "Cloud Cover (%)": data['CLOUD_AMT'][date],
            "Precipitation (mm)": data['PRECTOTCORR'][date]
        }
    else:
        return "No Weather Data Found"

# Function to get satellite overpass and weather data
def get_satellite_info(lat, lon, start_date, end_date):
    landsat_in_orbit = is_location_in_orbit(lon, lat, LANDSAT_SHAPEFILE)
    sentinel_in_orbit = is_location_in_orbit(lon, lat, SENTINEL_SHAPEFILE)
    
    if landsat_in_orbit or sentinel_in_orbit:
        overpass_times = get_satellite_overpass(lat, lon, start_date, end_date)
        weather_data = get_weather_data(lat, lon, start_date)
        return overpass_times, weather_data
    else:
        return "Location not in satellite orbit"

# Map and widgets for location selection and date input
def create_map_widget():
    # Initialize the map using ipyleaflet
    m = Map(center=(27.7, 85.3), zoom=10, basemap=basemap_to_tiles(basemaps.Esri.WorldImagery))

    # Marker to be added to the map
    marker = Marker(location=(27.7, 85.3))
    m.add_layer(marker)
    
    # Callback function for when the map is clicked
    def handle_click(event, **kwargs):
        lat, lon = event['coordinates']
        print(f"Selected location: Latitude: {lat}, Longitude: {lon}")
        # Update the lat/lon input fields
        lat_input.value = lat
        lon_input.value = lon
    
    # Callback function for when the marker is moved
    def handle_marker_move(event, **kwargs):
        lat, lon = event['coordinates']
        print(f"Marker moved: Latitude: {lat}, Longitude: {lon}")
        lat_input.value = lat
        lon_input.value = lon

    # Add a click event listener to the map using `on_interaction` instead of `on_click`
    m.on_interaction(handle_click)
    
    # Add event listener for marker move
    marker.on_move(handle_marker_move)

    return m

# Date pickers
start_date_picker = widgets.DatePicker(
    description='Start Date',
    value=datetime.date(2025, 3, 20),
    disabled=False
)
end_date_picker = widgets.DatePicker(
    description='End Date',
    value=datetime.date(2025, 3, 30),
    disabled=False
)

# Latitude and Longitude input fields
lat_input = widgets.FloatText(
    value=27.7, 
    description='Latitude:', 
    disabled=False
)

lon_input = widgets.FloatText(
    value=85.3, 
    description='Longitude:', 
    disabled=False
)

# Button to fetch the results
output = widgets.Output()

def on_button_click(b):
    with output:
        # Clear previous output
        output.clear_output()
        
        print("Fetching satellite info...")
        lat = lat_input.value
        lon = lon_input.value
        start_date = str(start_date_picker.value)
        end_date = str(end_date_picker.value)
        
        # Calling the function to get satellite and weather data
        result = get_satellite_info(lat, lon, start_date, end_date)
        
        if isinstance(result, tuple):
            overpass_times, weather_data = result
            print("Overpass Times:")
            print(overpass_times)
            print("Weather Data:")
            print(weather_data)
        else:
            print(result)

button = widgets.Button(description="Get Satellite and Weather Info")
button.on_click(on_button_click)

# Create map widget
map_widget = create_map_widget()

# Display everything
display(HBox([VBox([map_widget, lat_input, lon_input, start_date_picker, end_date_picker, button]), output]))


In [72]:
from skyfield.api import load, EarthSatellite
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

# Define TLE sources
tle_sources = {
    'LANDSAT 8': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2A': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2B': 'https://celestrak.org/NORAD/elements/resource.txt',
}

# Load satellites into a dictionary
satellites = {}
for name, url in tle_sources.items():
    satellite_list = load.tle_file(url)
    for sat in satellite_list:
        if name in sat.name:  # Ensure we are picking the correct satellite
            satellites[name] = sat
            print(f"Loaded TLE data for {name}")

# Verify loaded satellites
if not satellites:
    print("No satellites were loaded. Check TLE sources.")

# Load time scale
ts = load.timescale()

# Function to calculate precise overpass times using TLE
def find_overpasses(lat, lon, start_date, end_date, satellite):
    """Finds satellite overpass times when the nadir is closest to (lat, lon)."""
    satellite = satellites[satellite]
    start_dt = datetime.strptime(start_date, '%Y-%m-%d')
    end_dt = datetime.strptime(end_date, '%Y-%m-%d')

    results = []
    dt = start_dt

    while dt <= end_dt:
        # Generate time steps every 1 second for better accuracy
        t = ts.utc(dt.year, dt.month, dt.day, 0, 0, np.arange(0, 86400, 10))  # Every 310 seconds
        
        # Compute nadir position (ground track)
        subpoint = satellite.at(t).subpoint()
        latitudes = subpoint.latitude.degrees
        longitudes = subpoint.longitude.degrees

        # Find the closest approach to the given location
        distances = np.sqrt((latitudes - lat) ** 2 + (longitudes - lon) ** 2)
        min_index = np.argmin(distances)
        closest_time = t[min_index].utc_datetime()

        # Only store valid overpass times
        if distances[min_index] < 0.6:  # Threshold: 0.6° (~ km accuracy)
            results.append(closest_time.strftime('%Y-%m-%d %H:%M:%S'))
        
        # Move to the next possible overpass (repeat cycle)
        dt += timedelta(days=1)

    return results

# Function to get satellite overpass times
def get_precise_overpass(lat, lon, start_date, end_date):
    landsat_overpasses = find_overpasses(lat, lon, start_date, end_date, 'LANDSAT 8')
    sentinel_overpasses = find_overpasses(lat, lon, start_date, end_date, 'SENTINEL-2A')  # You can add SENTINEL-2B similarly
    
    landsat_data = pd.DataFrame({'date': landsat_overpasses, 'Satellite': 'Landsat 8'})
    sentinel_data = pd.DataFrame({'date': sentinel_overpasses, 'Satellite': 'Sentinel-2'})
    
    return pd.concat([landsat_data, sentinel_data], ignore_index=True)

# Example usage
lat = 27.7
lon = 85.3
start_date = '2025-05-30'
end_date = '2025-06-30'

overpass_times = get_precise_overpass(lat, lon, start_date, end_date)
print(overpass_times)


Loaded TLE data for LANDSAT 8
Loaded TLE data for SENTINEL-2A
Loaded TLE data for SENTINEL-2B
                  date   Satellite
0  2025-05-30 16:13:40   Landsat 8
1  2025-06-15 16:11:40   Landsat 8
2  2025-06-19 04:48:20   Landsat 8
3  2025-06-03 16:30:40  Sentinel-2
4  2025-06-13 16:30:10  Sentinel-2
5  2025-06-23 16:29:40  Sentinel-2
6  2025-06-30 05:09:00  Sentinel-2


In [69]:
from skyfield.api import load, Topos
from datetime import datetime, timedelta
import numpy as np
import pandas as pd

# Define TLE sources
tle_sources = {
    'LANDSAT 8': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2A': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2B': 'https://celestrak.org/NORAD/elements/resource.txt',
}

# Load satellites into a dictionary
satellites = {}
for name, url in tle_sources.items():
    satellite_list = load.tle_file(url)
    for sat in satellite_list:
        if name in sat.name:  # Ensure we are picking the correct satellite
            satellites[name] = sat
            print(f"Loaded TLE data for {name}")

# Verify loaded satellites
if not satellites:
    print("No satellites were loaded. Check TLE sources.")

# Load time scale and planetary ephemeris
ts = load.timescale()

# Function to calculate precise overpass times using TLE
def find_overpasses(lat, lon, start_date, end_date, satellite):
    """Finds satellite overpass times when the nadir is closest to (lat, lon)."""
    satellite = satellites[satellite]
    print(satellite)
    observer = Topos(latitude_degrees=lat, longitude_degrees=lon)
    print(observer)
    start_dt = datetime.strptime(start_date, '%Y-%m-%d')
    end_dt = datetime.strptime(end_date, '%Y-%m-%d')
    print(start_dt)

    results = []
    dt = start_dt
    count = 1

    while dt <= end_dt:
        # Generate time steps every 10 seconds for better accuracy
        t = ts.utc(dt.year, dt.month, dt.day, 0, 0, np.arange(0, 86400, 1))  # Every 10 seconds
        
        # Compute nadir position (ground track)
        subpoint = satellite.at(t).subpoint()
        latitudes = subpoint.latitude.degrees
        longitudes = subpoint.longitude.degrees

        # Find the closest approach to the given location
        distances = np.sqrt((latitudes - lat) ** 2 + (longitudes - lon) ** 2)
        
        min_index = np.argmin(distances)
        
        closest_time = t[min_index].utc_datetime()

        if count == 1:
            # np.set_printoptions(threshold=np.inf)
            print(f"subpoint: {subpoint}")
            # print(f"distances: {distances}")
            # print(f"min index: {min_index}")
        

        # Compute satellite position relative to observer
        topocentric = (satellite - observer).at(t[min_index])
        alt, az, distance = topocentric.altaz()

        # Prepare the output data
        if distances[min_index] < 0.5:  # Threshold: 0.6° (~ km accuracy)
            print(distances[min_index])
            results.append({
                'date': closest_time.strftime('%Y-%m-%d %H:%M:%S'),
                'Satellite': satellite.name,
                'Lat (DEG)': latitudes[min_index],
                'Lon (DEG)': longitudes[min_index],
                'Sat. Azi. (deg)': az.degrees,
                'Sat. Elev. (deg)': alt.degrees,
                'Range (km)': distance.km,
            })

        # Move to the next possible overpass (repeat cycle)
        dt += timedelta(days=1)
        count+=1

    return results

# Function to get satellite overpass times with additional parameters
def get_precise_overpass(lat, lon, start_date, end_date):
    landsat_overpasses = find_overpasses(lat, lon, start_date, end_date, 'LANDSAT 8')
    sentinel2A_overpasses = find_overpasses(lat, lon, start_date, end_date, 'SENTINEL-2A')  # You can add SENTINEL-2B similarly
    sentinel2B_overpasses = find_overpasses(lat, lon, start_date, end_date, 'SENTINEL-2B')
    
    landsat_data = pd.DataFrame(landsat_overpasses)
    sentinel2A_data = pd.DataFrame(sentinel2A_overpasses)
    sentinel2B_data = pd.DataFrame(sentinel2B_overpasses)
    
    return pd.concat([landsat_data, sentinel2A_data, sentinel2B_data], ignore_index=True)

# Example usage
lat = 27.7
lon = 85.3
start_date = '2025-05-30'
end_date = '2025-06-30'

overpass_times = get_precise_overpass(lat, lon, start_date, end_date)
print(overpass_times)


Loaded TLE data for LANDSAT 8
Loaded TLE data for SENTINEL-2A
Loaded TLE data for SENTINEL-2B
LANDSAT 8 catalog #39084 epoch 2025-03-31 02:33:30 UTC
IERS2010 latitude +27.7000 N longitude 85.3000 E elevation 0.0 m
2025-05-30 00:00:00
subpoint: IERS2010 latitude [+79.1830 +79.2226 ... -72.5422 -72.4891] N longitude [-7.5679e+01 -7.5928e+01 ... -5.2975e-02 -1.5175e-01] E elevation [716222.6 716226.2 ... 728181.2 728169.7] m
0.1733820572497892
0.27478842468620707
0.26209962247640883
SENTINEL-2A catalog #40697 epoch 2025-03-31 03:18:17 UTC
IERS2010 latitude +27.7000 N longitude 85.3000 E elevation 0.0 m
2025-05-30 00:00:00
subpoint: IERS2010 latitude [+13.7123 +13.7715 ... +57.1980 +57.1407] N longitude [-24.5612 -24.5748 ... 170.9806 170.9465] E elevation [791179.0 791181.0 ... 799139.3 799126.2] m
0.06136238233016587
0.15791631771848627
0.26725829670452744
SENTINEL-2B catalog #42063 epoch 2025-03-31 03:08:03 UTC
IERS2010 latitude +27.7000 N longitude 85.3000 E elevation 0.0 m
2025-05-30 

In [3]:
#compared this code with nasa prediction in table. 
from skyfield.api import load, EarthSatellite, Topos
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import requests

def fetch_tle_text(url):
    """Fetches TLE data from a given URL and returns the raw text."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        print(f"Error fetching TLE data from {url}: {e}")
        return ""

# Define TLE sources
tle_sources = {
    'LANDSAT 8': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2A': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2B': 'https://celestrak.org/NORAD/elements/resource.txt',
}

# Load satellites into a dictionary
satellites = {}
for name, url in tle_sources.items():
    satellite_list = load.tle_file(url)
    for sat in satellite_list:
        if name in sat.name:  # Ensure we are picking the correct satellite
            satellites[name] = sat
            print(f"Loaded TLE data for {name}")

# Verify loaded satellites
if not satellites:
    print("No satellites were loaded. Check TLE sources.")
# Load time scale and planetary ephemeris
ts = load.timescale()

# Function to calculate precise overpass times using TLE
def find_overpasses(lat, lon, start_date, end_date, satellite):
    """Finds satellite overpass times when the nadir is closest to (lat, lon)."""
    satellite = satellites[satellite]
    observer = Topos(latitude_degrees=lat, longitude_degrees=lon)
    
    start_dt = datetime.strptime(start_date, '%Y-%m-%d')
    end_dt = datetime.strptime(end_date, '%Y-%m-%d')

    results = []
    dt = start_dt

    while dt <= end_dt:
        # Generate time steps every 10 seconds for better accuracy
        t = ts.utc(dt.year, dt.month, dt.day, 0, 0, np.arange(0, 86400, 10))  # Every 10 seconds
        
        # Compute nadir position (ground track)
        geocentric = satellite.at(t)
        subpoint = geocentric.subpoint()
        latitudes = subpoint.latitude.degrees
        longitudes = subpoint.longitude.degrees
        
        # Use haversine formula for more accurate distance calculation
        lat_rad = np.radians(latitudes)
        lon_rad = np.radians(longitudes)
        lat_target_rad = np.radians(lat)
        lon_target_rad = np.radians(lon)
        
        dlat = lat_rad - lat_target_rad
        dlon = lon_rad - lon_target_rad
        a = np.sin(dlat / 2)**2 + np.cos(lat_target_rad) * np.cos(lat_rad) * np.sin(dlon / 2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
        distances = 6371 * c  # Earth radius in km
        
        min_index = np.argmin(distances)
        closest_time = t[min_index].utc_datetime()

        # Compute satellite position relative to observer
        topocentric = (satellite - observer).at(t[min_index])
        alt, az, distance = topocentric.altaz()

        # Prepare the output data
        if distances[min_index] < 50:  # Threshold: 50 km accuracy
            results.append({
                'date': closest_time.strftime('%Y-%m-%d %H:%M:%S'),
                'Satellite': satellite.name,
                'Lat (DEG)': latitudes[min_index],
                'Lon (DEG)': longitudes[min_index],
                'Sat. Azi. (deg)': az.degrees,
                'Sat. Elev. (deg)': alt.degrees,
                'Range (km)': distance.km,
            })

        # Move to the next possible overpass (repeat cycle)
        dt += timedelta(days=1)

    return results

# Function to get satellite overpass times with additional parameters
def get_precise_overpass(lat, lon, start_date, end_date):
    landsat_overpasses = find_overpasses(lat, lon, start_date, end_date, 'LANDSAT 8')
    sentinel2A_overpasses = find_overpasses(lat, lon, start_date, end_date, 'SENTINEL-2A')
    sentinel2B_overpasses = find_overpasses(lat, lon, start_date, end_date, 'SENTINEL-2B')
    
    landsat_data = pd.DataFrame(landsat_overpasses)
    sentinel2A_data = pd.DataFrame(sentinel2A_overpasses)
    sentinel2B_data = pd.DataFrame(sentinel2B_overpasses)
    
    return pd.concat([landsat_data, sentinel2A_data, sentinel2B_data], ignore_index=True)

# Example usage
lat = 27.7
lon = 85.3
start_date = '2025-05-30'
end_date = '2025-06-30'

overpass_times = get_precise_overpass(lat, lon, start_date, end_date)
print(overpass_times)

Loaded TLE data for LANDSAT 8
Loaded TLE data for SENTINEL-2A
Loaded TLE data for SENTINEL-2B
                  date    Satellite  Lat (DEG)  Lon (DEG)  Sat. Azi. (deg)  \
0  2025-05-30 16:13:40    LANDSAT 8  27.599900  85.146453       233.813569   
1  2025-06-15 16:11:40    LANDSAT 8  27.542492  85.622629       118.664899   
2  2025-06-19 04:48:20    LANDSAT 8  27.867978  85.072686       309.790292   
3  2025-06-03 16:30:40  SENTINEL-2A  27.839422  85.326513         9.589188   
4  2025-06-13 16:30:10  SENTINEL-2A  27.532154  85.504323       132.652026   
5  2025-06-23 16:29:40  SENTINEL-2A  27.484431  85.632494       125.981711   
6  2025-06-06 16:29:10  SENTINEL-2B  27.634729  85.715673        99.911293   
7  2025-06-13 05:08:30  SENTINEL-2B  27.820254  84.916172       289.501697   
8  2025-06-23 05:07:50  SENTINEL-2B  27.916394  85.099279       320.548109   

   Sat. Elev. (deg)  Range (km)  
0         88.307761  706.353632  
1         86.730747  707.320086  
2         87.372238  70

In [2]:
from skyfield.api import load, EarthSatellite, Topos
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import requests

def fetch_tle_text(url):
    """Fetches TLE data from a given URL and returns the raw text."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        print(f"Error fetching TLE data from {url}: {e}")
        return ""

# Define TLE sources
tle_sources = {
    'LANDSAT 8': 'https://celestrak.org/NORAD/elements/resource.txt',
    'LANDSAT 9': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2A': 'https://celestrak.org/NORAD/elements/resource.txt',  
    'SENTINEL-2B': 'https://celestrak.org/NORAD/elements/resource.txt',
    'SENTINEL-2C': None, # Manually provided TLE
}

# Manually provided TLE for SENTINEL-2C
sentinel_2c_tle = [
    "1 60989U 24157A   25090.79518797  .00000292  00000-0  12798-3 0  9993",
    "2 60989  98.5659 167.0180 0001050  95.0731 265.0572 14.30814009 29727"
]

today = datetime.today()
today.date();
print(today)

# Load time scale
ts = load.timescale()

# Load satellites into a dictionary
satellites = {}

for name, url in tle_sources.items():
    if name == 'SENTINEL-2C':
        line1, line2 = sentinel_2c_tle
        satellites[name] = EarthSatellite(line1, line2, name, ts)
        print(f"Loaded TLE data for {name}")
    else:
        tle_text = fetch_tle_text(url)
        tle_lines = tle_text.splitlines()
        for i in range(len(tle_lines) - 2):
            if name in tle_lines[i]:  # Look for the satellite name in the TLE file
                line1, line2 = tle_lines[i+1], tle_lines[i+2]
                satellites[name] = EarthSatellite(line1, line2, name, ts)
                print(f"Loaded TLE data for {name}")
                break

# Verify loaded satellites
if not satellites:
    print("No satellites were loaded. Check TLE sources.")

# Function to calculate precise overpass times using TLE
def find_overpasses(lat, lon, start_date, end_date, satellite):
    """Finds satellite overpass times when the nadir is closest to (lat, lon)."""
    if satellite not in satellites:
        print(f"Error: {satellite} TLE not available")
        return []

    sat = satellites[satellite]
    observer = Topos(latitude_degrees=lat, longitude_degrees=lon)
    
    start_dt = datetime.strptime(start_date, '%Y-%m-%d')
    end_dt = datetime.strptime(end_date, '%Y-%m-%d')

    results = []
    dt = start_dt

    while dt <= end_dt:
        # Generate time steps every 1 seconds for better accuracy
        t = ts.utc(dt.year, dt.month, dt.day, 0, 0, np.arange(0, 86400, 1))
        
        # Compute nadir position (ground track)
        subpoint = sat.at(t).subpoint()
        latitudes = subpoint.latitude.degrees
        longitudes = subpoint.longitude.degrees

        # Find the closest approach to the given location
        distances = np.sqrt((latitudes - lat) ** 2 + (longitudes - lon) ** 2)
        min_index = np.argmin(distances)
        closest_time = t[min_index].utc_datetime()

        # Compute satellite position relative to observer
        topocentric = (sat - observer).at(t[min_index])
        alt, az, distance = topocentric.altaz()

        # Prepare the output data
        if distances[min_index] < 0.7:  # Threshold: 0.5° (~ km accuracy)
            results.append({
                'date': closest_time.strftime('%Y-%m-%d %H:%M:%S'),
                'Satellite': satellite,
                'Lat (DEG)': latitudes[min_index],
                'Lon (DEG)': longitudes[min_index],
                'Sat. Azi. (deg)': az.degrees,
                'Sat. Elev. (deg)': alt.degrees,
                'Range (km)': distance.km,
            })

        # Move to the next possible overpass (repeat cycle)
        dt += timedelta(days=1)

    return results

# Function to get satellite overpass times with additional parameters
def get_precise_overpass(lat, lon, start_date, end_date):
    overpasses = []
    for sat_name in satellites.keys():
        overpasses.extend(find_overpasses(lat, lon, start_date, end_date, sat_name))
    
    return pd.DataFrame(overpasses)

# Example usage - salzburg hbf location
lat = 47.81306
lon = 13.04667
start_date = '2025-04-08'
end_date = '2025-06-08'

overpass_times = get_precise_overpass(lat, lon, start_date, end_date)
print(overpass_times)


daterange = f"prediction_{start_date}_to_{end_date}" 
overpass_times.to_csv(f"Outputs/prediction_{today}.csv", index=False)
print("Data added to CSV")


2025-04-12 00:17:31.864491
Loaded TLE data for LANDSAT 8
Loaded TLE data for LANDSAT 9
Loaded TLE data for SENTINEL-2A
Loaded TLE data for SENTINEL-2B
Loaded TLE data for SENTINEL-2C
                   date    Satellite  Lat (DEG)  Lon (DEG)  Sat. Azi. (deg)  \
0   2025-04-14 09:57:23    LANDSAT 8  47.932635  12.741232       300.314376   
1   2025-04-18 20:41:59    LANDSAT 8  47.936286  13.326226        56.656727   
2   2025-04-30 09:56:58    LANDSAT 8  47.889751  12.791781       294.186268   
3   2025-05-04 20:41:25    LANDSAT 8  47.951642  13.423368        61.179220   
4   2025-05-16 09:56:03    LANDSAT 8  47.839442  12.965740       295.866745   
5   2025-05-20 20:40:22    LANDSAT 8  48.038649  13.618908        59.387057   
6   2025-05-27 20:45:47    LANDSAT 8  47.557456  12.402619       239.786765   
7   2025-06-01 09:54:39    LANDSAT 8  47.717310  13.238322       126.476356   
8   2025-04-10 20:42:22    LANDSAT 9  47.912091  13.320774        61.660779   
9   2025-04-22 09:57:34    

OSError: [Errno 22] Invalid argument: 'Outputs/prediction_2025-04-12 00:17:31.864491.csv'